In [1]:
import pyomo.environ as pyo

In [2]:
# Create a model
model = pyo.ConcreteModel(name="Designer_Allocation")

model.I = pyo.Set(initialize=['Designer1', 'Designer2', 'Designer3'])
model.J = pyo.Set(initialize=['ProjectA', 'ProjectB', 'ProjectC', 'ProjectD'])

# Parameters
scores = {
    ('Designer1', 'ProjectA'): 90,
    ('Designer1', 'ProjectB'): 80,
    ('Designer1', 'ProjectC'): 10,
    ('Designer1', 'ProjectD'): 50,
    ('Designer2', 'ProjectA'): 60,
    ('Designer2', 'ProjectB'): 70,
    ('Designer2', 'ProjectC'): 50,
    ('Designer2', 'ProjectD'): 65,
    ('Designer3', 'ProjectA'): 70,
    ('Designer3', 'ProjectB'): 40,
    ('Designer3', 'ProjectC'): 80,
    ('Designer3', 'ProjectD'): 85,}

requirements = {
    'ProjectA': 70,
    'ProjectB': 50,
    'ProjectC': 85,
    'ProjectD': 35,}

availability = 80

model.x = pyo.Var(model.I, model.J, domain=pyo.NonNegativeReals)

# Objective: Maximize total score
def objective_rule(model):
    return sum(scores[i, j] * model.x[i, j] for i in model.I for j in model.J)
model.objective = pyo.Objective(rule=objective_rule, sense=pyo.maximize)

def supply_rule(model, i):
    return sum(model.x[i, j] for j in model.J) <= availability
model.supply_con = pyo.Constraint(model.I, rule=supply_rule)

def demand_rule(model, j):
    return sum(model.x[i, j] for i in model.I) >= requirements[j]
model.demand_con = pyo.Constraint(model.J, rule=demand_rule)

solver = pyo.SolverFactory('glpk')
results = solver.solve(model)

# Display results
print("Optimization Status:", results.solver.status)
print(f"Maximum Total Score: {pyo.value(model.objective)}")
print("\nAllocation (Hours):")
for i in model.I:
    for j in model.J:
        val = pyo.value(model.x[i, j])
        if val > 0:
            print(f"Designer {i} to Project {j}: {val} hours")

Optimization Status: ok
Maximum Total Score: 18825.0

Allocation (Hours):
Designer Designer1 to Project ProjectA: 70.0 hours
Designer Designer1 to Project ProjectB: 10.0 hours
Designer Designer2 to Project ProjectB: 40.0 hours
Designer Designer2 to Project ProjectC: 5.0 hours
Designer Designer2 to Project ProjectD: 35.0 hours
Designer Designer3 to Project ProjectC: 80.0 hours


In [3]:
import pyomo.environ as pyo

def solve_part_b():
    model = pyo.ConcreteModel()

    # 集合与数据
    nodes = [1, 2, 3, 4, 5, 6]
    costs = {1:40, 2:65, 3:43, 4:48, 5:72, 6:36}
    links = [(1,2), (1,4), (2,3), (2,5), (3,5), (3,6), (4,5), (5,6)]

    # 变量
    model.x = pyo.Var(nodes, domain=pyo.Binary)

    # 目标函数
    model.obj = pyo.Objective(expr=sum(costs[i] * model.x[i] for i in nodes), sense=pyo.minimize)

    # 约束：每条边必须被覆盖
    model.cover = pyo.ConstraintList()
    for (u, v) in links:
        model.cover.add(model.x[u] + model.x[v] >= 1)

    # 求解 (如果 glpk 不在路径中，请参考之前的建议指定 executable 路径)
    solver = pyo.SolverFactory('glpk')
    solver.solve(model)

    print("--- Part (b) Results ---")
    print(f"Minimum Total Cost: {pyo.value(model.obj)}")
    selected = [i for i in nodes if pyo.value(model.x[i]) > 0.5]
    print(f"Stations at: {selected}")

solve_part_b()

--- Part (b) Results ---
Minimum Total Cost: 155.0
Stations at: [1, 3, 5]
